# DSA 2050 | Week 2 Practical Lab
## SQL, Data Acquisition and Join Validation

**Student:** Tamara Mwakisha  
**Student ID:** 660146  
**Currency:** Kenyan shillings (`Ksh.`)

This practical examines customer and order data at their respective grains, tests relational key quality before integration, and validates joins using row counts and sales control totals. The source anomalies are retained for diagnosis before the prescribed customer-key clean-up is applied. Results are reported at order grain unless otherwise stated.

## A. Create the source data
The following cells reproduce the supplied lab data without pre-emptively removing the duplicated customer key or the unmatched order. They write `customers.csv`, `orders.csv`, and `regions.json` beside this notebook.

In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

customers = pd.DataFrame({
    "CustomerID": ["C001", "C002", "C003", "C004", "C004", "C005", "C006", "C007", "C008", "C009", "C010"],
    "CustomerName": ["Amina", "Brian", "Carol", "David", "David", "Esther", "Faith", "George", "Halima", "Ian", "Jane"],
    "Segment": ["Retail", "Corporate", "Retail", "SME", "SME", "Corporate", "SME", "Retail", "Corporate", "Retail", "SME"],
    "RegionCode": ["NRB", "MSA", "NRB", "KSM", "KSM", "NKR", "NRB", "MSA", "KSM", "NKR", "NRB"]
})

orders = pd.DataFrame({
    "OrderID": [f"O{i:03d}" for i in range(1, 16)],
    "CustomerID": ["C001", "C002", "C001", "C003", "C004", "C005", "C006", "C002", "C007", "C008", "C009", "C010", "C006", "C003", "C999"],
    "OrderAmount": [4500, 12500, 3200, 6800, 9100, 15000, 7600, 8300, 2500, 11000, 5400, 6200, 4300, 7200, 9900],
    "ProductCategory": ["Electronics", "Office", "Home", "Electronics", "Office", "Electronics", "Home", "Office", "Home", "Electronics", "Office", "Home", "Electronics", "Office", "Electronics"]
})

regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]

customers.to_csv(ROOT / "customers.csv", index=False)
orders.to_csv(ROOT / "orders.csv", index=False)
with open(ROOT / "regions.json", "w", encoding="utf-8") as file:
    json.dump(regions, file, indent=4)

print("Wrote customers.csv, orders.csv and regions.json to the notebook directory.")

Wrote customers.csv, orders.csv and regions.json to the notebook directory.


## B. Inspect the sources
Reloading the saved files makes the persisted source data—not only in-memory objects—the basis for the analysis.

In [2]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

source_profile = pd.DataFrame([
    {"Source": "customers.csv", "Rows": len(customers), "Columns": len(customers.columns), "Intended grain": "One customer record per CustomerID"},
    {"Source": "orders.csv", "Rows": len(orders), "Columns": len(orders.columns), "Intended grain": "One transaction per OrderID"}
])
display(source_profile)
print("Customer sample")
display(customers.head())
print("Order sample")
display(orders.head())

,Source,Rows,Columns,Intended grain
0,customers.csv,11,4,One customer record per CustomerID
1,orders.csv,15,4,One transaction per OrderID


Customer sample


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Order sample


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


**Q1.** One row in `customers.csv` is intended to represent one customer. That intended grain is not met by the raw file: `C004` is recorded twice.

**Q2.** One row in `orders.csv` represents one order, identified by `OrderID`.

**Q3.** The datasets are not at the same grain. The customer file is customer-level; the order file is transaction-level, with a customer potentially associated with several orders. Joining them should therefore be many-to-one from orders to customers.

## C. Detect key problems
Assess uniqueness and referential coverage before joining. The source records remain unchanged at this stage.

In [3]:
duplicate_count = int(customers["CustomerID"].duplicated().sum())
duplicate_customers = customers.loc[
    customers["CustomerID"].duplicated(keep=False)
].sort_values(["CustomerID", "CustomerName"])

customer_ids = set(customers["CustomerID"])
order_customer_ids = set(orders["CustomerID"])
unmatched_customer_ids = sorted(order_customer_ids - customer_ids)

print(f"Repeated CustomerID values after first occurrence: {duplicate_count}")
display(duplicate_customers)
print("CustomerIDs present in orders but absent from customers:", unmatched_customer_ids)

Repeated CustomerID values after first occurrence: 1


,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


CustomerIDs present in orders but absent from customers: ['C999']


**Q4.** A duplicate `CustomerID` on the customer side violates the intended one-row-per-customer key. Every order for that key can match multiple customer rows, multiplying transactions and overstating counts and sales.

**Q5.** `C999` is present in the orders but absent from the customer file.

**Q6.** An unmatched key may signal an incomplete customer extract, a recording error, or a valid transaction awaiting identification. Deleting it without investigation would conceal the exception and remove an order worth `Ksh. 9,900` from the analysis.

## D. Create the SQLite database
Load both original tables into SQLite. The database is recreated on each run so that execution from a fresh kernel is deterministic.

In [4]:
conn = sqlite3.connect("retail_lab.db")
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

test_query = "SELECT * FROM customers LIMIT 5;"
database_sample = pd.read_sql_query(test_query, conn)
assert len(database_sample) == 5
assert {"customers", "orders"}.issubset(
    set(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)["name"])
)
display(database_sample)
print("SQLite tables verified: customers, orders.")

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


SQLite tables verified: customers, orders.


## E. `SELECT` and `WHERE`
The following statements use SQL for the requested retrieval and filtering tasks. Monetary amounts are shown in `Ksh.`; the stored `OrderAmount` values remain numeric.

In [5]:
# E1 — Return only the specified order fields.
select_query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders
ORDER BY OrderID;
"""
selected_orders = pd.read_sql_query(select_query, conn)
display(selected_orders.assign(OrderAmount=selected_orders["OrderAmount"].map(lambda x: f"Ksh. {x:,.0f}")))

# E2 — Apply each lab filter independently.
retail_query = "SELECT CustomerID, CustomerName, Segment, RegionCode FROM customers WHERE Segment = 'Retail' ORDER BY CustomerID;"
over_8000_query = "SELECT OrderID, CustomerID, OrderAmount, ProductCategory FROM orders WHERE OrderAmount > 8000 ORDER BY OrderID;"
electronics_query = "SELECT OrderID, CustomerID, OrderAmount FROM orders WHERE ProductCategory = 'Electronics' ORDER BY OrderID;"
electronics_over_8000_query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders
WHERE ProductCategory = 'Electronics' AND OrderAmount > 8000
ORDER BY OrderID;
"""

retail_result = pd.read_sql_query(retail_query, conn)
over_8000_result = pd.read_sql_query(over_8000_query, conn)
electronics_result = pd.read_sql_query(electronics_query, conn)
electronics_over_8000_result = pd.read_sql_query(electronics_over_8000_query, conn)

def display_money(df, columns):
    shown = df.copy()
    for column in columns:
        shown[column] = shown[column].map(lambda value: f"Ksh. {value:,.0f}")
    display(shown)

print("Retail-segment customers")
display(retail_result)
print("Orders above Ksh. 8,000")
display_money(over_8000_result, ["OrderAmount"])
print("Electronics orders")
display_money(electronics_result, ["OrderAmount"])
print("Electronics orders above Ksh. 8,000")
display_money(electronics_over_8000_result, ["OrderAmount"])

,OrderID,CustomerID,OrderAmount
0,O001,C001,"Ksh. 4,500"
1,O002,C002,"Ksh. 12,500"
2,O003,C001,"Ksh. 3,200"
3,O004,C003,"Ksh. 6,800"
4,O005,C004,"Ksh. 9,100"
5,O006,C005,"Ksh. 15,000"
6,O007,C006,"Ksh. 7,600"
7,O008,C002,"Ksh. 8,300"
8,O009,C007,"Ksh. 2,500"
9,O010,C008,"Ksh. 11,000"


Retail-segment customers


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


Orders above Ksh. 8,000


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,"Ksh. 12,500",Office
1,O005,C004,"Ksh. 9,100",Office
2,O006,C005,"Ksh. 15,000",Electronics
3,O008,C002,"Ksh. 8,300",Office
4,O010,C008,"Ksh. 11,000",Electronics
5,O015,C999,"Ksh. 9,900",Electronics


Electronics orders


,OrderID,CustomerID,OrderAmount
0,O001,C001,"Ksh. 4,500"
1,O004,C003,"Ksh. 6,800"
2,O006,C005,"Ksh. 15,000"
3,O010,C008,"Ksh. 11,000"
4,O013,C006,"Ksh. 4,300"
5,O015,C999,"Ksh. 9,900"


Electronics orders above Ksh. 8,000


,OrderID,CustomerID,OrderAmount
0,O006,C005,"Ksh. 15,000"
1,O010,C008,"Ksh. 11,000"
2,O015,C999,"Ksh. 9,900"


## F. `GROUP BY` and aggregate functions
Category performance is aggregated directly from the order table. `COUNT(*)` counts transactions, `SUM` measures booked sales, and `AVG` reports mean order value.

In [6]:
category_query = """
SELECT ProductCategory AS "Product Category",
       COUNT(*) AS "Number of Orders",
       SUM(OrderAmount) AS total_sales,
       AVG(OrderAmount) AS average_order_value
FROM orders
GROUP BY ProductCategory
ORDER BY CASE ProductCategory
           WHEN 'Electronics' THEN 1
           WHEN 'Home' THEN 2
           WHEN 'Office' THEN 3
           ELSE 4
         END;
"""
category_kpis_raw = pd.read_sql_query(category_query, conn)
category_kpis = category_kpis_raw.rename(columns={
    "total_sales": "Total Sales (Ksh.)",
    "average_order_value": "Average Order Value (Ksh.)"
}).copy()
category_kpis["Total Sales (Ksh.)"] = category_kpis["Total Sales (Ksh.)"].map(lambda x: f"Ksh. {x:,.0f}")
category_kpis["Average Order Value (Ksh.)"] = category_kpis["Average Order Value (Ksh.)"].map(lambda x: f"Ksh. {x:,.4f}")
display(category_kpis)

,Product Category,Number of Orders,Total Sales (Ksh.),Average Order Value (Ksh.)
0,Electronics,6,"Ksh. 51,500","Ksh. 8,583.3333"
1,Home,4,"Ksh. 19,500","Ksh. 4,875.0000"
2,Office,5,"Ksh. 42,500","Ksh. 8,500.0000"


**Q7. Observations.**

1. Electronics led category revenue at `Ksh. 51,500` from 6 orders; its mean order value was `Ksh. 8,583.3333`. It generated about 45.4% of total recorded order value (`Ksh. 113,500`).
2. Home had the lowest revenue (`Ksh. 19,500`) and average order value (`Ksh. 4,875.0000`). Office recorded `Ksh. 42,500` across 5 orders, with an average (`Ksh. 8,500.0000`) close to Electronics despite one fewer order. These are revenue and order-value comparisons only; margin and product cost are unavailable.

## G. Join customers and orders
Run the prescribed `LEFT JOIN` against the original customer table. This deliberately exposes the effect of the non-unique key while retaining orders without a customer match.

In [7]:
join_query = """
SELECT o.OrderID,
       o.CustomerID,
       c.CustomerName,
       c.Segment,
       o.OrderAmount
FROM orders AS o
LEFT JOIN customers AS c
       ON o.CustomerID = c.CustomerID
ORDER BY o.OrderID, c.CustomerName;
"""
joined = pd.read_sql_query(join_query, conn)
join_count_check = pd.DataFrame([
    {"Measure": "Order rows before join", "Value": len(orders)},
    {"Measure": "Rows after original LEFT JOIN", "Value": len(joined)},
    {"Measure": "Additional rows from join", "Value": len(joined) - len(orders)}
])
display(join_count_check)
joined_display = joined.copy()
joined_display["OrderAmount"] = joined_display["OrderAmount"].map(lambda x: f"Ksh. {x:,.0f}")
display(joined_display)

,Measure,Value
0,Order rows before join,15
1,Rows after original LEFT JOIN,16
2,Additional rows from join,1


,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,"Ksh. 4,500"
1,O002,C002,Brian,Corporate,"Ksh. 12,500"
2,O003,C001,Amina,Retail,"Ksh. 3,200"
3,O004,C003,Carol,Retail,"Ksh. 6,800"
4,O005,C004,David,SME,"Ksh. 9,100"
5,O005,C004,David,SME,"Ksh. 9,100"
6,O006,C005,Esther,Corporate,"Ksh. 15,000"
7,O007,C006,Faith,SME,"Ksh. 7,600"
8,O008,C002,Brian,Corporate,"Ksh. 8,300"
9,O009,C007,George,Retail,"Ksh. 2,500"


**Q8.** No. There are 15 orders before joining and 16 output rows afterwards. The excess row is caused by `C004`: the order table has one O005 transaction for this ID, but the customer table has two matching rows, so O005 is returned twice. The left join retains C999 as an unmatched row.

**Q9.** For O015 / C999, `CustomerName` and `Segment` are NULL (displayed as `NaN` in the pandas result).

## H. Reconcile totals
Sales are a control total at order grain. Comparing totals before and after the join identifies the monetary consequence of row multiplication.

In [8]:
source_sales = int(orders["OrderAmount"].sum())
joined_sales = int(joined["OrderAmount"].sum())
reconciliation = pd.DataFrame([
    {"Stage": "Orders before join", "Rows": len(orders), "Sales (Ksh.)": source_sales},
    {"Stage": "After original LEFT JOIN", "Rows": len(joined), "Sales (Ksh.)": joined_sales},
    {"Stage": "Change", "Rows": len(joined) - len(orders), "Sales (Ksh.)": joined_sales - source_sales}
])
reconciliation["Sales (Ksh.)"] = reconciliation["Sales (Ksh.)"].map(lambda x: f"Ksh. {x:,.0f}")
display(reconciliation)

,Stage,Rows,Sales (Ksh.)
0,Orders before join,15,"Ksh. 113,500"
1,After original LEFT JOIN,16,"Ksh. 122,600"
2,Change,1,"Ksh. 9,100"


**Q10.** No. Sales increase from `Ksh. 113,500` before the join to `Ksh. 122,600` after it.

**Q11.** `C004` causes the increase: O005 (`Ksh. 9,100`) appears twice because the customer key occurs twice.

**Q12.** Row counts and control totals test whether integration has preserved the transaction population and its value. Without reconciliation, a syntactically valid join can silently duplicate or omit records and produce materially incorrect management KPIs.

## I. Correct the duplicate-key problem
The lab directs a one-row-per-ID customer table for the second join. The two source rows for C004 agree on the available descriptive fields; retain the first source occurrence, while keeping the raw `customers` table intact for auditability. The unmatched order is not removed.

In [9]:
customers_clean = customers.drop_duplicates(subset="CustomerID", keep="first").copy()
assert customers_clean["CustomerID"].is_unique
customers_clean.to_sql("customers_clean", conn, if_exists="replace", index=False)

clean_join_query = """
SELECT o.OrderID,
       o.CustomerID,
       c.CustomerName,
       c.Segment,
       o.OrderAmount
FROM orders AS o
LEFT JOIN customers_clean AS c
       ON o.CustomerID = c.CustomerID
ORDER BY o.OrderID;
"""
clean_joined = pd.read_sql_query(clean_join_query, conn)
clean_rows = len(clean_joined)
clean_sales = int(clean_joined["OrderAmount"].sum())
clean_reconciliation = pd.DataFrame([
    {"Validation": "Order row count", "Source orders": len(orders), "Clean-join result": clean_rows, "Pass": clean_rows == len(orders)},
    {"Validation": "Sales total (Ksh.)", "Source orders": f"Ksh. {source_sales:,.0f}", "Clean-join result": f"Ksh. {clean_sales:,.0f}", "Pass": clean_sales == source_sales}
])
display(clean_reconciliation)
assert clean_rows == len(orders)
assert clean_sales == source_sales

,Validation,Source orders,Clean-join result,Pass
0,Order row count,15,15,True
1,Sales total (Ksh.),"Ksh. 113,500","Ksh. 113,500",True


**Q13.** Yes. The cleaned left join returns 15 rows and `Ksh. 113,500`, equal to the source row count and sales total. The corrected key restores one customer match per ID; C999 remains unmatched but its order and value remain in the result.

## J. Customer segment KPIs
Aggregate the cleaned customer-order relationship by segment. A NULL segment is explicitly labelled so that the unassigned transaction is visible rather than absorbed into a known segment.

In [10]:
segment_query = """
SELECT COALESCE(c.Segment, 'Unmatched / NULL') AS Segment,
       COUNT(*) AS Orders,
       SUM(o.OrderAmount) AS total_sales,
       AVG(o.OrderAmount) AS average_order_value
FROM orders AS o
LEFT JOIN customers_clean AS c
       ON o.CustomerID = c.CustomerID
GROUP BY c.Segment
ORDER BY CASE WHEN c.Segment IS NULL THEN 4
              WHEN c.Segment = 'Corporate' THEN 1
              WHEN c.Segment = 'Retail' THEN 2
              WHEN c.Segment = 'SME' THEN 3
              ELSE 5 END;
"""
segment_kpis_raw = pd.read_sql_query(segment_query, conn)
segment_kpis = segment_kpis_raw.rename(columns={
    "total_sales": "Total Sales (Ksh.)",
    "average_order_value": "Average Order Value (Ksh.)"
}).copy()
segment_kpis["Total Sales (Ksh.)"] = segment_kpis["Total Sales (Ksh.)"].map(lambda x: f"Ksh. {x:,.0f}")
segment_kpis["Average Order Value (Ksh.)"] = segment_kpis["Average Order Value (Ksh.)"].map(lambda x: f"Ksh. {x:,.4f}")
display(segment_kpis)

,Segment,Orders,Total Sales (Ksh.),Average Order Value (Ksh.)
0,Corporate,4,"Ksh. 46,800","Ksh. 11,700.0000"
1,Retail,6,"Ksh. 29,600","Ksh. 4,933.3333"
2,SME,4,"Ksh. 27,200","Ksh. 6,800.0000"
3,Unmatched / NULL,1,"Ksh. 9,900","Ksh. 9,900.0000"


**Q14.** Corporate accounts for the highest attributed segment sales (`Ksh. 46,800` across 4 orders) and the highest average order value (`Ksh. 11,700.0000`). Retail has the largest known-segment order count (6) and `Ksh. 29,600` in sales, but the lowest known-segment average order value (`Ksh. 4,933.3333`). SME contributes `Ksh. 27,200` from 4 orders, while C999 remains unclassified (`Ksh. 9,900`); management should resolve that customer reference before treating the segment split as complete.

## K. Merge the JSON region lookup
Load the lookup as a separate source and merge it to the cleaned customer table. `validate="many_to_one"` confirms that each customer code resolves to at most one lookup row.

In [11]:
regions_df = pd.read_json("regions.json")
customers_regions = customers_clean.merge(
    regions_df,
    on="RegionCode",
    how="left",
    validate="many_to_one"
)
display(customers_regions[["CustomerID", "CustomerName", "Segment", "RegionCode", "RegionName"]])
assert customers_regions["RegionName"].notna().all()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru
5,C006,Faith,SME,NRB,Nairobi
6,C007,George,Retail,MSA,Mombasa
7,C008,Halima,Corporate,KSM,Kisumu
8,C009,Ian,Retail,NKR,Nakuru
9,C010,Jane,SME,NRB,Nairobi


**Q15.** Confirmed. Each of the 10 distinct customer IDs in `customers_clean` has a readable `RegionName`: Nairobi, Mombasa, Kisumu, or Nakuru.

**Q16.** With a left merge, an unrecognised RegionCode would not remove the customer row; its `RegionName` would be missing (`NaN`/NULL). That exception should be investigated.

## Final challenge — Regional sales analysis
Attach the region lookup to the cleaned order join and aggregate at order grain. The unassigned C999 order is retained as a separate row so the reported regional totals reconcile to total sales.

In [12]:
order_region = orders.merge(
    customers_clean[["CustomerID", "RegionCode"]],
    on="CustomerID",
    how="left",
    validate="many_to_one"
).merge(
    regions_df,
    on="RegionCode",
    how="left",
    validate="many_to_one"
)
order_region["Region"] = order_region["RegionName"].fillna("Unmatched / NULL")
regional_order = ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Unmatched / NULL"]
regional_kpis_raw = (order_region
    .groupby("Region", as_index=False, dropna=False)
    .agg(NumberOfOrders=("OrderID", "count"), TotalSales=("OrderAmount", "sum")))
regional_kpis_raw["Region"] = pd.Categorical(regional_kpis_raw["Region"], categories=regional_order, ordered=True)
regional_kpis_raw = regional_kpis_raw.sort_values("Region")
regional_kpis = regional_kpis_raw.copy()
regional_kpis["Total Sales (Ksh.)"] = regional_kpis["TotalSales"].map(lambda x: f"Ksh. {x:,.0f}")
regional_kpis = regional_kpis.rename(columns={"NumberOfOrders": "Number of Orders"})[["Region", "Number of Orders", "Total Sales (Ksh.)"]]
display(regional_kpis)
assert int(regional_kpis_raw["TotalSales"].sum()) == source_sales

,Region,Number of Orders,Total Sales (Ksh.)
2,Nairobi,7,"Ksh. 39,800"
1,Mombasa,3,"Ksh. 23,300"
0,Kisumu,2,"Ksh. 20,100"
3,Nakuru,2,"Ksh. 20,400"
4,Unmatched / NULL,1,"Ksh. 9,900"


**Q17.** Nairobi has the highest sales among the four identified regions: 7 orders totalling `Ksh. 39,800`.

**Q18.** Management should not equate the highest revenue with the best overall performance. First, the data contains no product or regional costs, so gross margin and profit cannot be compared. Second, the extract covers only 15 orders and has no market-size, customer-base, or time-period context; it cannot show whether the sales level is high relative to opportunity or sustainable. In addition, one order (`Ksh. 9,900`) has no customer/region match, so the regional allocation is incomplete.

## Short conclusion
The duplicated C004 key was the principal data-quality issue: it caused the original left join to duplicate O005 and overstate sales by `Ksh. 9,100`. After the duplicate was documented and one matching customer row retained, the join returned 15 orders and reconciled to the source total of `Ksh. 113,500`; the unmatched C999 order remained visible. Such reconciliation is necessary because valid SQL syntax does not establish that a join preserves grain or totals. Corporate generated the highest attributed segment sales at `Ksh. 46,800`. The analysis is limited by the small extract, absent cost and market context, and the unresolved region for C999.

## Reproducibility and submission notes
Run the notebook from top to bottom after restarting the kernel. It recreates the supplied CSV/JSON sources and `retail_lab.db` in the notebook directory, performs the SQL queries, and asserts key reconciliation checks. The repository contains the notebook, three source files, and this README; the generated database is reproducible and need not be versioned.